# 02, Transformação de Dados (Camada Silver)

## Objetivo
Realizar a limpeza e a padronização dos dados brutos de internações
hospitalares do SUS em Minas Gerais (competência 2024), transformando
a camada Bronze em dados confiáveis e consultáveis.

## Operações realizadas
1. Deduplicação pela chave natural do SIH (competência + N_AIH).
2. Conversão das datas de internação e saída para o tipo date.
3. Filtro do escopo: internações ocorridas em 2024.
4. Tipagem correta das colunas (valores monetários e códigos numéricos).
5. Verificação de nulos e gravação em formato Delta.

In [0]:
# Leitura da camada Bronze, gravada em Parquet pelo notebook de coleta.
# A Bronze contém os registros brutos de MG/2024 e os metadados de
# controle adicionados na ingestão (dt_ingestao, fonte, uf, ano_competencia).
from pyspark.sql.functions import col, count, when

caminho_bronze = "/Volumes/workspace/default/dados_mvp/bronze/sih_rd_mg_2024"
caminho_silver = "/Volumes/workspace/default/dados_mvp/silver/sih_rd_mg_2024"

df_bronze = spark.read.parquet(caminho_bronze)

print("Linhas na Bronze:", df_bronze.count())
print("Colunas:", len(df_bronze.columns))

# Análise de completude: contagem de valores nulos nas colunas-chave.
# O resultado esperado é zero nulos, conforme validado na camada Bronze.
colunas_chave = ["N_AIH", "ANO_CMPT", "MES_CMPT", "DT_INTER", "DT_SAIDA",
                 "DIAG_PRINC", "SEXO", "IDADE"]
df_bronze.select([count(when(col(c).isNull(), c)).alias(c) for c in colunas_chave]).show()

In [0]:
# Etapa 1: deduplicação pela chave natural (ANO_CMPT + MES_CMPT + N_AIH).
# O arquivo RD do DATASUS contém repetições de AIH reprocessadas; a
# primeira ocorrência de cada chave é mantida. A remoção esperada é de
# aproximadamente 193 linhas.
from pyspark.sql.functions import to_date, col

df_silver = df_bronze.dropDuplicates(["ANO_CMPT", "MES_CMPT", "N_AIH"])
print("Após deduplicação:", df_silver.count())

# Etapa 2: conversão das datas de internação e saída do formato string
# YYYYMMDD para o tipo date, o que permite cálculos de permanência e
# séries temporais.
df_silver = (df_silver
    .withColumn("DT_INTER", to_date(col("DT_INTER"), "yyyyMMdd"))
    .withColumn("DT_SAIDA", to_date(col("DT_SAIDA"), "yyyyMMdd")))

# Etapa 3: filtro do escopo de análise, restringindo às internações
# ocorridas em 2024. A competência 2024 inclui reprocessamentos de
# internações de anos anteriores, que ficam fora do escopo do trabalho.
df_silver = df_silver.filter((col("DT_INTER") >= "2024-01-01") &
                             (col("DT_INTER") <= "2024-12-31"))
print("Após filtro (internações em 2024):", df_silver.count())

In [0]:
# Etapa 4: tipagem das colunas. Valores monetários convertidos para
# double e códigos numéricos para int. A conversão é condicional, pois
# o leiaute do arquivo RD varia entre versões.
from pyspark.sql.functions import col, count, when

colunas_valor = ["VAL_SH", "VAL_SP", "VAL_SADT", "VAL_TOT", "VAL_CF", "VAL_ST"]
for c in colunas_valor:
    if c in df_silver.columns:
        df_silver = df_silver.withColumn(c, col(c).cast("double"))

colunas_codigo = ["ANO_CMPT", "MES_CMPT", "QT_DIARIAS", "IDADE"]
for c in colunas_codigo:
    if c in df_silver.columns:
        df_silver = df_silver.withColumn(c, col(c).cast("int"))

# Etapa 5: verificação de nulos após as transformações. Valores nulos em
# DT_SAIDA podem indicar internação em curso no encerramento do período.
# A decisão é preservar o registro e tratar o caso na etapa de análise.
print("Nulos restantes nas colunas-chave:")
df_silver.select([count(when(col(c).isNull(), c)).alias(c)
                  for c in ["DT_SAIDA", "DIAG_PRINC", "SEXO"]]).show()

In [0]:
# Etapa 6: gravação da camada Silver em formato Delta.
# A pasta de destino é removida antes da gravação para garantir
# idempotência total: cada execução recria a camada do zero com o
# schema atual. Isso também elimina o erro de incompatibilidade de
# metadados quando o schema do DataFrame muda entre execuções
# (ex: inclusão das colunas uf e ano_competencia na Bronze).
dbutils.fs.rm(caminho_silver, recurse=True)

df_silver.write.mode("overwrite").format("delta").save(caminho_silver)
print("Silver salva em:", caminho_silver)

# Verificação final: total de registros e unicidade da chave natural.
df_check = spark.read.format("delta").load(caminho_silver)
print("TOTAL na Silver:", df_check.count())

duplicadas = (df_check
    .groupBy("ANO_CMPT", "MES_CMPT", "N_AIH")
    .count()
    .filter("count > 1"))
print("Duplicatas na Silver:", duplicadas.count())

In [0]:
# Diagnóstico da coluna IDADE e SEXO na Silver
df_silver = spark.read.format("delta").load(caminho_silver)

df_silver.select("IDADE").printSchema()
df_silver.groupBy("IDADE").count().orderBy("count", ascending=False).show(20, truncate=False)
df_silver.groupBy("SEXO").count().show()

In [0]:
from pyspark.sql import DataFrame as SparkDF

# Caminho da camada Silver no volume (formato Delta)
caminho_silver = "/Volumes/workspace/default/dados_mvp/silver/sih_rd_mg_2024"
catalogo = spark.catalog.currentCatalog()
schema = spark.catalog.currentDatabase()
nome_tabela = "silver_sih_rd_mg_2024"

# Detecta automaticamente qual DataFrame da sessão é a camada Silver
candidatos = {}
for nome, obj in list(globals().items()):
    if isinstance(obj, SparkDF) and not any(p in nome.lower() for p in ["bronze", "gold"]):
        candidatos[nome] = obj

nome_preferido = None
for nome in ["df_silver", "silver_df", "df_limpo", "silver", "df"]:
    if nome in candidatos:
        nome_preferido = nome
        break

if nome_preferido is None:
    if not candidatos:
        raise SystemExit("Nenhum DataFrame Spark encontrado. Rode as células anteriores do 02_silver.")
    nome_preferido = list(candidatos.keys())[-1]

df_silver = candidatos[nome_preferido]
print("DataFrame usado como Silver:", nome_preferido, "| linhas:", df_silver.count())

# 1. Persiste no volume em Delta (idempotente, não acumula dados)
df_silver.write.mode("overwrite").format("delta").save(caminho_silver)
print("Silver persistida em Delta:", caminho_silver)

# 2. Registra no catálogo como tabela gerenciada, removendo a versão antiga
# para evitar o erro de merge de schema (DELTA_FAILED_TO_MERGE_FIELDS)
spark.sql(f"DROP TABLE IF EXISTS {catalogo}.{schema}.{nome_tabela}")
df_silver.write.mode("overwrite").saveAsTable(f"{catalogo}.{schema}.{nome_tabela}")

# 3. Descrição da tabela no catálogo (contexto + linhagem, como exige a seção 4.3)
spark.sql(f"COMMENT ON TABLE {catalogo}.{schema}.{nome_tabela} IS 'Camada Silver do pipeline: internacoes hospitalares do SIH/SUS (RD) de Minas Gerais, competencia 2024, limpas e padronizadas. Origem: DATASUS/SIH via PySUS (12 arquivos mensais RDMG2401 a RDMG2412). Transformacoes: remocao de duplicatas por competencia + N_AIH, padronizacao de tipos e datas, adicao de metadados de controle.'")

# 4. Descrição das colunas (nome + descrição + tipo + domínio + linhagem)
descricoes = {
    "N_AIH": "Numero da AIH, identificador da internacao. Texto. Origem: campo N_AIH do arquivo RD.",
    "ANO_CMPT": "Ano de competencia do faturamento. Inteiro. Domínio: 2024. Origem: campo ANO_CMPT do arquivo RD.",
    "MES_CMPT": "Mes de competencia do faturamento. Inteiro. Domínio: 1 a 12. Origem: campo MES_CMPT do arquivo RD.",
    "DT_INTER": "Data da internacao. Data (YYYYMMDD). Origem: campo DT_INTER do arquivo RD.",
    "DT_SAIDA": "Data da saida. Data (YYYYMMDD). Origem: campo DT_SAIDA do arquivo RD.",
    "DIAG_PRINC": "Diagnostico principal, codigo CID-10. Texto. Domínio: codigos validos CID-10. Origem: campo DIAG_PRINC do arquivo RD.",
    "SEXO": "Sexo do paciente. Inteiro. Domínio: 1 = Masculino, 3 = Feminino (leiaute SIH). Origem: campo SEXO do arquivo RD.",
    "IDADE": "Idade em anos. Inteiro. Domínio: 0 a 120. Origem: campo IDADE do arquivo RD, convertido pelo PySUS.",
    "MORTE": "Indicador de obito. Inteiro. Domínio: 0 = nao, 1 = sim. Origem: campo MORTE do arquivo RD.",
    "MUNIC_RES": "Codigo IBGE do municipio de residencia. Texto. Domínio: codigos IBGE de 6 digitos. Origem: campo MUNIC_RES do arquivo RD.",
    "CGC_HOSP": "CNPJ do hospital. Texto. Origem: campo CGC_HOSP do arquivo RD.",
    "QT_DIARIAS": "Quantidade de diarias. Inteiro. Origem: campo QT_DIARIAS do arquivo RD.",
    "VAL_TOT": "Valor total da AIH. Decimal. Origem: campo VAL_TOT do arquivo RD.",
    "dt_ingestao": "Data e hora da ingestao na camada Bronze. Timestamp. Adicionado no pipeline de coleta.",
    "fonte": "Fonte dos dados. Texto. Valor fixo: DATASUS/SIH. Adicionado no pipeline de coleta.",
    "uf": "Unidade federativa. Texto. Valor fixo: MG. Adicionado no pipeline de coleta.",
    "ano_competencia": "Ano de competencia. Inteiro. Valor fixo: 2024. Adicionado no pipeline de coleta.",
}
for coluna, descricao in descricoes.items():
    spark.sql(f"COMMENT ON COLUMN {catalogo}.{schema}.{nome_tabela}.{coluna} IS '{descricao}'")

# 5. Exibe o describe table no formato Catalog Explorer, com as descrições
spark.sql(f"DESCRIBE TABLE EXTENDED {catalogo}.{schema}.{nome_tabela}").show(truncate=False)

# 6. Evidência da persistência no volume
print("Arquivos na pasta da Silver:")
display(dbutils.fs.ls(caminho_silver))